<a href="https://colab.research.google.com/github/Yahimeen/IA_AplicadaLLama_Hackathon_01/blob/main/Hackathon%20Asistente%20con%20Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **  HACKATHON -- ASISTENTE VIRTUAL **

Este Colab implementa en un solo flujo lo visto en los tres Temas principales: inferencia básica, RAG y fine-tuning con LoRa para construir un asistente que recupera contexto propio y responde con un modelo ya ajustado a un tono específico.

Para este asistente virtual se usa informacion de mi curriculum para proporcionar informacion inexistente en los modelos preentrenados para poder validar las respuestas.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

**Abriendo la sesion usando el token de Hugging Face**

In [15]:
# Instalar librerías e iniciar sesión en Hugging Face con el token desde Colab Secrets

# Remover los comentarios cuando se instala la primera vez
!pip install transformers peft accelerate trl sentence-transformers --quiet
!pip install PyMuPDF --quiet
!pip install chromadb --quiet
!pip install groq -q

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
from groq import Groq

logging.set_verbosity_error()

CHUNCK_SIZE=500
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
modelo_pequeno = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

pdf_file_path = "Resume Pedro Ramirez - Associated Director Digital Business Solutions - GT.pdf"

login(token=userdata.get('DEV_AI_READ_ONLY'))
print("Sesión de Hugging Face iniciada correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.4 MB/s eta 0:00:00
Sesión de Hugging Face iniciada correctamente.


Empezaremos por definir el base line del asistente, confirmando que usando solamente el conocimiento del modelo, no puede responder a las preguntas.

In [19]:
#
def asistente_virtual_sincontexto(pregunta):

    prompt = f"""Responde la pregunta.

              Pregunta: {pregunta}

              Proporciona una respuesta Concisa, no incluyas caracteres extraños en la respuesta."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b", # Usar el modelo Groq para la generación
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

pregunta = "¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?"

print(f"Pregunta: {pregunta}")
print(f"Respuesta: {asistente_virtual_sincontexto(pregunta)}\n")

Pregunta: ¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?
Respuesta: No dispongo de información sobre la experiencia de Pedro Ramírez en gestión de proyectos.



## **PASO 1: LA BASE DE CONOCIMIENTO (RAG)**

Primero vamos a crear una funcion para extraer el texto del archivo que usaremos como base de conocimiento.

Indexaremos esta base de conocimientos con sentence-transformer para poder utilizarla para recuperar el fragmento mas relevante para responder la pregunta y evitar alucionaciones al responder.



In [6]:
import pymupdf
import fitz

def extract_pdf_chunks(pdf_path, chunk_size=1000):
    """
      Esta funcion extrae el texto de un archivo y genera 'chunks' de tamaño definido por el usuario, el default es 1000 caracteres.
    """
    chunks_list = []
    try:
        document = fitz.open(pdf_path)
        current_text = ""
        for page_num in range(document.page_count):
            page = document.load_page(page_num)
            page_text = page.get_text()
            current_text += page_text

            # Agrega los chunks a una lista
            while len(current_text) >= chunk_size:
                chunks_list.append(current_text[:chunk_size])
                current_text = current_text[chunk_size:]

        # Agrega el texto faltante a la lista
        if current_text:
            chunks_list.append(current_text)

        document.close()
        return chunks_list
    except FileNotFoundError:
        print(f"Error: Archivo PDF no encontrado en : {pdf_path}")
        return []
    except Exception as e:
        print(f"Ocurrio un error: {e}")
        return []


Luego vamos a extraer la informacion del archivo que sera nuestra base de datos de conocimientos para el asistente.

In [9]:

all_chunks = list(extract_pdf_chunks(pdf_file_path, chunk_size=CHUNCK_SIZE))

print(f"Total chunks extracted: {len(all_chunks)}")
for i, chunk in enumerate(all_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print("\n")

Total chunks extracted: 29
--- Chunk 1 ---
Pedro Enrique Ramirez Lopez
90 S Regan Mead Cir, The Woodlands, TX, 77382
+1 (203) 300 2267 – yahimeen@gmail.com
Admissions Committee, Online Master of Science in Computer Science (OMSCS)
I am pleased to submit my application to the Online Master of Science in Computer Science (OMSCS) program
at the Georgia Institute of Technology. With more than two decades of experience in information
technology, software development, data analytics, automation, and digital transformation, I am seeking to
furt


--- Chunk 2 ---
her strengthen my computer science foundation while expanding my expertise in artificial intelligence
and advanced computing technologies.
Throughout my career, I have worked at the intersection of business and technology, leading initiatives
involving software development, data engineering, analytics, robotic process automation, process mining,
conversational AI, machine learning, and generative AI. In my current role leading Digital


Luego vamos a usar los chucks para crear los embeddings.


In [10]:
# Este codigo deberia ser de una sola vez para poder generar los embeddings

from sentence_transformers import SentenceTransformer
import numpy as np

modelo_embeddings = SentenceTransformer(model_name)
print("Modelo de embeddings cargado:", modelo_embeddings)

embeddings_documentos = modelo_embeddings.encode(all_chunks)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo de embeddings cargado: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)
Embeddings generados: (29, 384)



A continuacion, usaremos una base de datos vectorial (chromadb) para almacenar los embeddings.

Definimos una funcion para almacenar los embeddings, de manera similar a la generacion de los embeddings, esta parte del proyecto se deberia llamar solamente cada vez que se requiera actualizar la base de conocimientos.


In [11]:
import chromadb

def guardar_embeddings_en_chromadb(fragmentos, embeddings, nombre_coleccion='fragmentos_curriculum'):
    """
    Almacena fragmentos y sus embeddings en una colección de ChromaDB.
    """
    cliente = chromadb.Client()
    # Crea una nueva colección u obtiene una existente
    coleccion = cliente.get_or_create_collection(name=nombre_coleccion)

    # Genera IDs para cada fragmento (pueden ser números secuenciales simples)
    ids_fragmentos = [f"fragmento_{i}" for i in range(len(fragmentos))]

    coleccion.add(
        embeddings=embeddings.tolist(), # ChromaDB espera una lista de listas
        documents=fragmentos,
        ids=ids_fragmentos
    )
    print(f"Almacenados {len(fragmentos)} fragmentos en la colección de ChromaDB '{nombre_coleccion}'.")
    return coleccion

# Llama a la función para almacenar tus embeddings generados
coleccion_chroma = guardar_embeddings_en_chromadb(all_chunks, embeddings_documentos)
print(f"Número de elementos en la colección: {coleccion_chroma.count()}")

Almacenados 29 fragmentos en la colección de ChromaDB 'fragmentos_curriculum'.
Número de elementos en la colección: 29



Ahora que los embeddings están almacenados en ChromaDB, vamos a usar una funcion para hacer busquedas en la colección por similitud.


In [12]:
# Funcion para hacer busquedas en la colección de ChromaDB
def consultar_coleccion_chroma(texto_consulta, coleccion, n_resultados=1):
    embedding_consulta = modelo_embeddings.encode([texto_consulta]).tolist()
    resultados = coleccion.query(
        query_embeddings=embedding_consulta,
        n_results=n_resultados,
        include=['documents', 'distances']
    )
    return resultados

# Prueba la función de consulta
consulta = "¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?"
resultados_consulta = consultar_coleccion_chroma(consulta, coleccion_chroma, n_resultados=3)

print(f"Consulta: {consulta}")
print("Mejores resultados de ChromaDB:")
for i, (doc, dist) in enumerate(zip(resultados_consulta['documents'][0], resultados_consulta['distances'][0])):
    print(f"--- Resultado {i+1} (Distancia: {dist:.4f}) ---")
    print(doc)
    print("\n")

Consulta: ¿Cuál es la experiencia de Pedro en gestión de proyectos?
Mejores resultados de ChromaDB:
--- Resultado 1 (Distancia: 16.4337) ---
oblem Solving
Research-Oriented Technology
Evaluation
Technical Documentation and
Executive Communication
Project Management and Cross-
Functional Collaboration
Responsible AI, Governance,
Compliance, and Ethics
Experience
Associate Director - Digital Business Solutions Americas
July 2021 – Present
Academic-Relevant Experience and Technical Contributions:

Annual Budget Planning for the DBS team: Collaborated in the preparation and management of
the annual budget for the Digital Business Soluti


--- Resultado 2 (Distancia: 17.1772) ---
e efficient query resolution for
business stakeholders.

Strategic Technology Leadership: Defined the two-year strategic roadmap for the integration of
Process Mining, RPAs, Conversational AI, and other digital technologies based on evolving business
demands. Orchestrated initiatives that aligned IT systems and t

## **PASO 2: AJUSTAR EL MODELO (FINE-TUNING CON LoRA)**

De acuerdo a los parametros solicitados para este proyecto, a continuacion se usara un modelo ligero basado en Llama para ajustarlo con LoRA para que responda siempre en el mismo tono breve y directo.
Usaremos el mismo modelo sugerido en el Colab de la sesion correspondiente.

In [13]:
# Cargar el modelo base y su tokenizer

tokenizer = AutoTokenizer.from_pretrained(modelo_pequeno)
modelo = AutoModelForCausalLM.from_pretrained(modelo_pequeno, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_pequeno)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=60):
    mensajes = [{"role": "user", "content": pregunta}]
    prompt_formateado = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(prompt_formateado, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()

prompt_prueba = "¿Puedo renovar un préstamo?"

In [ ]:
# Definir el dataset de ejemplos y convertirlo en Dataset

from datasets import Dataset

def formatear_ejemplo(pregunta, respuesta):
    mensajes = [
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return tokenizer.apply_chat_template(mensajes, tokenize=False)

pares = [
    ("¿Puedo renovar un préstamo?", "Sí, puedes renovarlo una vez si nadie más lo ha solicitado."),
    ("¿Hasta cuándo puedo tener un libro prestado?", "El préstamo estándar es de 15 días."),
    ("¿Necesito credencial para entrar a la biblioteca?", "Sí, es necesario mostrar tu credencial vigente en la entrada."),
    ("¿Puedo reservar una sala de estudio?", "Sí, puedes reservarla hasta con 2 días de anticipación desde el portal."),
    ("¿Hay wifi disponible dentro de la biblioteca?", "Sí, la red 'Biblioteca-Invitados' está disponible en todas las salas."),
]

ejemplos = [{"texto": formatear_ejemplo(p, r)} for p, r in pares]
dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

In [ ]:
# Probar el modelo base con el prompt de prueba antes de ajustarlo

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

Sí, puedes renovar el préstemo si deseas. La renovación de un préstergo es una operación de reembolso de la cantidad de la inversión original, pero con una nueva fecha de retención. En este caso, debes


In [ ]:
# Configurar LoRA (semilla fija para un resultado reproducible en la grabación)

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# Entrenar con LoRA

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados_pipeline",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = trainer.state.log_history[-2]['loss'] # último paso del entrenamiento
#perdida_final = resultado_entrenamiento.training_loss # promedio del entrenamiento
print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '0.1527', 'grad_norm': '1.13', 'learning_rate': '0.0002', 'entropy': '0.4406', 'num_tokens': '251', 'mean_token_accuracy': '0.9634', 'epoch': '1'}
{'loss': '0.1094', 'grad_norm': '1.515', 'learning_rate': '0.0001933', 'entropy': '0.3626', 'num_tokens': '502', 'mean_token_accuracy': '0.9715', 'epoch': '2'}
{'loss': '0.1002', 'grad_norm': '2.077', 'learning_rate': '0.0001867', 'entropy': '0.3107', 'num_tokens': '753', 'mean_token_accuracy': '0.9797', 'epoch': '3'}
{'loss': '0.07859', 'grad_norm': '1.207', 'learning_rate': '0.00018', 'entropy': '0.2774', 'num_tokens': '1004', 'mean_token_accuracy': '0.9797', 'epoch': '4'}
{'loss': '0.065', 'grad_norm': '0.8116', 'learning_rate': '0.0001733', 'entropy': '0.2234', 'num_tokens': '1255', 'mean_token_accuracy': '0.9797', 'epoch': '5'}
{'loss': '0.05889', 'grad_norm': '0.9098', 'learning_rate': '0.0001667', 'entropy': '0.1873', 'num_tokens': '1506', 'mean_token_accuracy': '0.9797', 'epoch': '6'}
{'loss': '0.0544', 'grad_norm': '0.9158'

## **PASO 3: DESPLEGAR EL MODELO AJUSTADO**

"Desplegar" en este contexto significa guardar el adaptador LoRA de forma reutilizable, no levantar un servidor. Con `save_pretrained` queda listo para volver a cargarlo en cualquier notebook sin repetir el entrenamiento; `push_to_hub` es opcional si quieres tenerlo disponible en tu cuenta de Hugging Face.

In [ ]:
# Guardar el adaptador LoRA localmente

modelo_lora.save_pretrained("/content/modelo_ajustado_lora")
print("Adaptador LoRA guardado en /content/modelo_ajustado_lora")

# Opcional: subir el adaptador a tu cuenta de Hugging Face para reutilizarlo fuera de esta sesión
# modelo_lora.push_to_hub("tu-usuario/tinyllama-atencion-clientes-lora")

Adaptador LoRA guardado en /content/modelo_ajustado_lora


## **PASO 4: VERIFICACION DE RESPUESTAS CON RAG Y CON EL MODELO AJUSTADO**

Con la base de conocimiento indexada y el modelo ya ajustado, conectamos ambas piezas en una sola función: recupera el fragmento relevante, se lo entrega al modelo ajustado junto con la pregunta, y genera la respuesta final. Este es el mismo patrón que se espera construir en el Hackathon 1.

Para validar la respuesta, usaremos la misma pregunta que se hizo en las primeras celdas.

In [20]:
def asistente_virtual_con_chromadb(pregunta):
    # Recuperar el fragmento más relevante de ChromaDB
    resultados_recuperados = consultar_coleccion_chroma(pregunta, coleccion_chroma, n_resultados=1)
    fragmento = resultados_recuperados['documents'][0][0] # Tomar el primer y más relevante fragmento

    prompt = f"""Responde la pregunta usando SOLO la informacion proporcionada a continuacion.
                 Si la informacion no responde la pregunta, no inventes ninguna respuesta, responde diciendo que no sabes la respuesta.

Informacion: {fragmento}

Pregunta: {pregunta}

Respuesta Concisa, sin caracteres extraños."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b", # Usar el modelo Groq para la generación
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def ejecutar_demostracion_asistente(preguntas_ejemplo):
    for pregunta in preguntas_ejemplo:
        print(f"Pregunta: {pregunta}")
        print(f"Respuesta: {asistente_virtual_con_chromadb(pregunta)}\n")

pregunta = "¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?"

print(f"Pregunta: {pregunta}")
print(f"Respuesta: {asistente_virtual_sincontexto(pregunta)}\n")

Pregunta: ¿Cuál es la experiencia de Pedro Ramirez en gestión de proyectos?
Respuesta: Pedro Ramírez tiene más de 8 años de experiencia liderando proyectos de tecnología y transformación digital, con certificaciones PMP y Scrum Master, y ha entregado con éxito más de 30 iniciativas en empresas multinacionales.



# **CONCLUSIONES: **

Se observa que usando el metodo RAG que los resultados son inmediatos y se eliminan las aluciones.

Esto se debe a que, con el modelo LoRA se necesitan realizar un entrenamiento con un mayor numero de casos para poder afectar el comportamiento del modelo.
